<a href="https://colab.research.google.com/github/yutaota/intro_to_causal_inference/blob/main/Ex03_RCT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1. データ生成

In [ ]:
# 利用するライブラリのインポート
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 潜在結果変数データの生成
import numpy as np

n = 4000
causal_effect = 5

np.random.seed(42)
u = np.random.randn(n) # 重症度
Y0 = u
Y1 = Y0 + causal_effect + (0.5 * u)

# 処置変数と観測可能データの作成
propensity_scores = 1 / (1 + np.exp(- (u))) # 重症度が高いほど良く治療を受ける
# A = np.random.binomial(1, propensity_scores, n)
A = np.random.binomial(1, 0.5, n) # ランダム割当

Y = A * Y1 + (1 - A) * Y0 # スイッチング方程式

data = pd.DataFrame({'Y': Y, 'A': A})

## 2. 処置効果の推定

In [ ]:
# 回帰
model = sm.OLS(Y, sm.add_constant(A))
results = model.fit()
print(results.summary())
print(f"")

# 単純比較
mean_y_given_a1 = Y[A == 1].mean()
mean_y_given_a0 = Y[A == 0].mean()
mean_difference = mean_y_given_a1 - mean_y_given_a0
print(f"SDO: {mean_difference}")

## 3. 仮定の確認

In [ ]:
# 無相関の仮定
covariance_ua = np.cov(u, A)[0, 1]
print(f"---無相関の仮定---")
print(f"Covariance between A and u: {covariance_ua}")
print(f"")

# 平均独立の仮定
mean_y1 = Y1.mean()
mean_y1_given_a1 = Y1[A == 1].mean()
print(f"---平均独立の仮定---")
print(f"Mean of Y1 given A=1: {mean_y1_given_a1}")
print(f"Mean of Y1: {mean_y1}")
print(f"")

mean_y0_given_a0 = Y0[A == 0].mean()
mean_y0 = Y0.mean()
print(f"Mean of Y0 given A=0: {mean_y0_given_a0}")
print(f"Mean of Y0: {mean_y0}")
print(f"")

# 選択バイアスと異質性バイアスの確認
# Y1, Y0は観測できないので、観測可能なデータからは確認できない。

# 選択バイアス
mean_y0_given_a1 = Y0[A == 1].mean()
mean_y0_given_a0 = Y0[A == 0].mean()
print(f"---選択バイアス---")
print(f"Mean of Y0 given A=1: {mean_y0_given_a1}")
print(f"Mean of Y0 given A=0: {mean_y0_given_a0}")
print(f"")

# 異質性バイアス
effect = Y1 - Y0
mean_effect_given_a1 = effect[A == 1].mean()
mean_effect_given_a0 = effect[A == 0].mean()
print(f"---異質性バイアス---")
print(f"Mean of effect given A=1: {mean_effect_given_a1}")
print(f"Mean of effect given A=0: {mean_effect_given_a0}")